In [ ]:
import pandas as pd
from sqlalchemy import create_engine, inspect
import urllib.parse
from io import StringIO
import json

In [ ]:
# ==========================================
# 1. CARREGAR CONFIGURAÇÕES E DEFINIR DADOS
# ==========================================

# Lê as credenciais do arquivo JSON
with open('db_config.json', 'r') as f:
    DB_CONFIG = json.load(f)

DADOS_BRUTOS = """
1;Linus Torvals
2;Richard Stallman
3;Guido van Rossum
4;Bill Gates
5;Mark Zuckerberg
"""

In [ ]:
# ==========================================
# 2. FUNÇÕES MODULARIZADAS
# ==========================================

def processar_dados_pandas(dados_texto):
    df = pd.read_csv(StringIO(dados_texto), sep=';', names=['id', 'nome'])
    return df

def criar_engine_conexao(config):
    senha_codificada = urllib.parse.quote_plus(config['password'])
    string_conexao = (
        f"mssql+pyodbc://{config['user']}:{senha_codificada}"
        f"@{config['server']}:{config['port']}/{config['database']}"
        "?driver=ODBC+Driver+17+for+SQL+Server"
    )
    return create_engine(string_conexao)

def carregar_dados_sql_pandas(df, engine):
    inspetor = inspect(engine)
    
    if inspetor.has_table('Usuarios'):
        df_existente = pd.read_sql('SELECT id FROM tb_usuarios', engine)
        ids_existentes = df_existente['id'].tolist()
        df_novos = df[~df['id'].isin(ids_existentes)]
    else:
        df_novos = df
        
    if not df_novos.empty:
        df_novos.to_sql('tb_usuarios', con=engine, if_exists='append', index=False)
        return df_novos
    else:
        return pd.DataFrame()

In [ ]:
# ==========================================
# 3. EXECUÇÃO PRINCIPAL
# ==========================================
def main():
    try:
        print("Processando dados com Pandas...")
        df_dados = processar_dados_pandas(DADOS_BRUTOS)
        
        engine = criar_engine_conexao(DB_CONFIG)
        
        print("Realizando carga no SQL Server...")
        df_inseridos = carregar_dados_sql_pandas(df_dados, engine)
        
        print("\n--- RESUMO DA CARGA DE DADOS ---")
        if not df_inseridos.empty:
            print(f"Total de {len(df_inseridos)} registro(s) inserido(s) com sucesso:")
            for index, row in df_inseridos.iterrows():
                print(f" -> ID: {row['id']} | Nome: {row['nome']}")
        else:
            print("Nenhum dado novo foi inserido. Todos os registros já constavam no banco.")
            
    except Exception as e:
        print(f"Erro inesperado: {e}")

In [ ]:
# Executar a função principal
main()